In [29]:
import requests
import pandas as pd
import time
from pathlib import Path

In [30]:
#1.setting
sy=2015
ey=2025
yrs=list(range(sy,ey+1))

ctys={"MYS":{"name":"Malaysia","is_benchmark":False},
    "SGP":{"name":"Singapore","is_benchmark":False},
    "THA":{"name":"Thailand","is_benchmark":False},
    "IDN":{"name":"Indonesia","is_benchmark":False},
    "VNM":{"name":"Vietnam","is_benchmark":False},
    "PHL":{"name":"Philippines","is_benchmark":False},
    "KHM":{"name":"Cambodia","is_benchmark":False},
    "LAO":{"name":"Laos","is_benchmark":False},
    "CHN":{"name":"China","is_benchmark":True}}

inds={"SL.IND.EMPL.ZS":{"name":"Employment in industry (% of total employment)","grp":"EVI"},
    "SL.AGR.EMPL.ZS":{"name":"Employment in agriculture (% of total employment)","grp":"EVI"},
    "SL.SRV.EMPL.ZS":{"name":"Employment in services (% of total employment)","grp":"EVI"},
    "SL.UEM.TOTL.ZS":{"name":"Unemployment,total (% of total labor force)","grp":"EVI"},
    "SL.UEM.1524.ZS":{"name":"Youth unemployment (% of total labor force ages 15-24)","grp":"EVI"},
    "IT.NET.USER.ZS":{"name":"Individuals using the Internet (% of population)","grp":"DRI"},
    "IT.CEL.SETS.P2":{"name":"Mobile cellular subscriptions (per 100 people)","grp":"DRI"},
    "IT.NET.BBND.P2":{"name":"Fixed broadband subscriptions (per 100 people)","grp":"DRI"}}

Path("raw-data").mkdir(exist_ok=True)
Path("outputs").mkdir(exist_ok=True)
print("Settings loaded")
print("country number:",len(ctys))
print("indicator number:",len(inds))
print("year range:",sy,"-",ey)

Settings loaded
country number: 9
indicator number: 8
year range: 2015 - 2025


In [31]:
#2.download data
all_rows=[]
fail_rows=[]
task=0
total=len(ctys)*len(inds)

for cc in ctys:
    for ind in inds:
        task=task+1
        print(f"[{task}/{total}] downloading {cc}-{ind}")
        url=f"https://api.worldbank.org/v2/country/{cc}/indicator/{ind}?format=json&date={sy}:{ey}&per_page=1000"
        ok=False
        err_msg=""

        for try_id in range(1,4):
            try:
                rsp=requests.get(url,timeout=30)

                if rsp.status_code==200:
                    data=rsp.json()

                    if isinstance(data,list) and len(data)>=2:
                        ok=True
                        break
                    else:
                        err_msg="invalid json structure"
                else:
                    err_msg=f"status code {rsp.status_code}"

            except Exception as e:
                err_msg=str(e)
            print(f"retry {try_id}/3 failed:",cc,ind,err_msg)
            time.sleep(try_id)

        if ok==False:
            fail_rows.append({"country_code":cc,
                "country_name":ctys[cc]["name"],
                "is_benchmark":ctys[cc]["is_benchmark"],
                "indicator_code":ind,
                "indicator_name":inds[ind]["name"],
                "indicator_group":inds[ind]["grp"],
                "error":err_msg,"url":url})
            continue

        if data[1] is None:
            continue

        for item in data[1]:
            all_rows.append({"country_code":cc,
                "country_name":ctys[cc]["name"],
                "is_benchmark":ctys[cc]["is_benchmark"],
                "indicator_code":ind,
                "indicator_name":inds[ind]["name"],
                "indicator_group":inds[ind]["grp"],
                "year":int(item["date"]),
                "value":item["value"],
                "source":"World Bank API"})
        time.sleep(0.2)

raw_df=pd.DataFrame(all_rows)
fail_df=pd.DataFrame(fail_rows)
print("download finished")
print("raw shape:",raw_df.shape)
print("failed requests:",fail_df.shape[0])

display(raw_df.head(20))

if fail_df.shape[0]>0:
    display(fail_df)
    fail_path=f"outputs/worldbank_failed_requests_{sy}_{ey}.csv"
    fail_df.to_csv(fail_path,index=False,encoding="utf-8-sig")
    print("failed request file saved:",fail_path)
else:
    print("no failed api requests")

[1/72] downloading MYS-SL.IND.EMPL.ZS
[2/72] downloading MYS-SL.AGR.EMPL.ZS
[3/72] downloading MYS-SL.SRV.EMPL.ZS
[4/72] downloading MYS-SL.UEM.TOTL.ZS
[5/72] downloading MYS-SL.UEM.1524.ZS
[6/72] downloading MYS-IT.NET.USER.ZS
[7/72] downloading MYS-IT.CEL.SETS.P2
[8/72] downloading MYS-IT.NET.BBND.P2
[9/72] downloading SGP-SL.IND.EMPL.ZS
[10/72] downloading SGP-SL.AGR.EMPL.ZS
[11/72] downloading SGP-SL.SRV.EMPL.ZS
[12/72] downloading SGP-SL.UEM.TOTL.ZS
[13/72] downloading SGP-SL.UEM.1524.ZS
[14/72] downloading SGP-IT.NET.USER.ZS
[15/72] downloading SGP-IT.CEL.SETS.P2
[16/72] downloading SGP-IT.NET.BBND.P2
[17/72] downloading THA-SL.IND.EMPL.ZS
[18/72] downloading THA-SL.AGR.EMPL.ZS
[19/72] downloading THA-SL.SRV.EMPL.ZS
[20/72] downloading THA-SL.UEM.TOTL.ZS
[21/72] downloading THA-SL.UEM.1524.ZS
[22/72] downloading THA-IT.NET.USER.ZS
[23/72] downloading THA-IT.CEL.SETS.P2
[24/72] downloading THA-IT.NET.BBND.P2
[25/72] downloading IDN-SL.IND.EMPL.ZS
[26/72] downloading IDN-SL.AGR.EMP

,country_code,country_name,is_benchmark,indicator_code,indicator_name,indicator_group,year,value,source
0,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2025,25.955851,World Bank API
1,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2024,26.002224,World Bank API
2,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2023,25.996687,World Bank API
3,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2022,26.071842,World Bank API
4,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2021,25.929687,World Bank API
5,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2020,26.166868,World Bank API
6,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2019,27.924213,World Bank API
7,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2018,27.111039,World Bank API
8,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2017,27.717453,World Bank API
9,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2016,27.486797,World Bank API


no failed api requests


In [32]:
#3.check raw data
print("years:")
print(sorted(raw_df["year"].unique()))
print("countries:")
print(raw_df["country_code"].unique())
print("indicators:")
print(raw_df["indicator_code"].unique())

# check duplicate rows in raw data
dup_df=raw_df[raw_df.duplicated(
        subset=["country_code","indicator_code","year"],
        keep=False)].copy()
print("duplicate rows:",dup_df.shape[0])

if dup_df.shape[0]>0:
    display(dup_df.sort_values(["country_code","indicator_code","year"]))
else:
    print("no duplicate rows")

raw_path=f"raw-data/worldbank_raw_api_data_{sy}_{ey}.csv"
raw_df.to_csv(raw_path,index=False,encoding="utf-8-sig")
print("raw data saved:",raw_path)

years:
[np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
countries:
['MYS' 'SGP' 'THA' 'IDN' 'VNM' 'PHL' 'KHM' 'LAO' 'CHN']
indicators:
['SL.IND.EMPL.ZS' 'SL.AGR.EMPL.ZS' 'SL.SRV.EMPL.ZS' 'SL.UEM.TOTL.ZS'
 'SL.UEM.1524.ZS' 'IT.NET.USER.ZS' 'IT.CEL.SETS.P2' 'IT.NET.BBND.P2']
duplicate rows: 0
no duplicate rows
raw data saved: raw-data/worldbank_raw_api_data_2015_2025.csv


In [33]:
#4.create full grid
grid_rows=[]
for cc in ctys:
    for ind in inds:
        for y in yrs:
            grid_rows.append({
                "country_code":cc,
                "country_name":ctys[cc]["name"],
                "is_benchmark":ctys[cc]["is_benchmark"],
                "indicator_code":ind,
                "indicator_name":inds[ind]["name"],
                "indicator_group":inds[ind]["grp"],
                "year":y})

grid_df=pd.DataFrame(grid_rows)
print("grid shape:",grid_df.shape)
display(grid_df.head())

exp_grid=len(ctys)*len(inds)*len(yrs)
act_grid=grid_df.shape[0]
print("expected grid rows:",exp_grid)
print("actual grid rows:",act_grid)
if act_grid==exp_grid:
    print("grid row number is correct")
else:
    print("check needed:grid row number mismatch")

grid shape: (792, 7)


,country_code,country_name,is_benchmark,indicator_code,indicator_name,indicator_group,year
0,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2015
1,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2016
2,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2017
3,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2018
4,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2019


expected grid rows: 792
actual grid rows: 792
grid row number is correct


In [34]:
#5.merge panel
panel_df=grid_df.merge(
    raw_df[["country_code","country_name","is_benchmark","indicator_code",
        "indicator_name","indicator_group","year","value","source"]],
    on=["country_code","country_name","is_benchmark","indicator_code",
        "indicator_name","indicator_group","year"],how="left")

panel_df["value"]=pd.to_numeric(panel_df["value"],errors="coerce")
panel_df["is_missing"]=panel_df["value"].isna()
print("panel shape:",panel_df.shape)
display(panel_df.head(20))
exp_panel=len(ctys)*len(inds)*len(yrs)
act_panel=panel_df.shape[0]
print("expected panel rows:",exp_panel)
print("actual panel rows:",act_panel)

if act_panel==exp_panel:
    print("panel row number is correct")
else:
    print("check needed: panel row number mismatch")
panel_path=f"outputs/worldbank_panel_{sy}_{ey}.csv"
panel_df.to_csv(panel_path,index=False,encoding="utf-8-sig")
print("panel data saved:",panel_path)

panel shape: (792, 10)


,country_code,country_name,is_benchmark,indicator_code,indicator_name,indicator_group,year,value,source,is_missing
0,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2015,27.518841,World Bank API,False
1,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2016,27.486797,World Bank API,False
2,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2017,27.717453,World Bank API,False
3,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2018,27.111039,World Bank API,False
4,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2019,27.924213,World Bank API,False
5,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2020,26.166868,World Bank API,False
6,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2021,25.929687,World Bank API,False
7,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2022,26.071842,World Bank API,False
8,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2023,25.996687,World Bank API,False
9,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2024,26.002224,World Bank API,False


expected panel rows: 792
actual panel rows: 792
panel row number is correct
panel data saved: outputs/worldbank_panel_2015_2025.csv


In [35]:
#6.coverage by country and indicator
grp_cols=["country_code","country_name","is_benchmark",
    "indicator_code","indicator_name","indicator_group"]

sum_rows=[]
for key,g in panel_df.groupby(grp_cols):
    cc,cname,bench,ind,iname,igrp=key
    miss=list(g.loc[g["is_missing"],"year"].astype(int))
    avail=list(g.loc[~g["is_missing"],"year"].astype(int))

    total_y=len(g)
    avail_y=len(avail)
    miss_y=len(miss)

    sum_rows.append({"country_code":cc,"country_name":cname,"is_benchmark":bench,
        "indicator_code":ind,"indicator_name":iname,"indicator_group":igrp,
        "total_years":total_y,"available_years":avail_y,"missing_years_count":miss_y,
        "missing_years":miss,"first_available_year":min(avail) if len(avail)>0 else None,
        "latest_available_year":max(avail) if len(avail)>0 else None,
        "coverage_rate":avail_y/total_y})

cov_df=pd.DataFrame(sum_rows)
cov_df=cov_df.sort_values(
    ["is_benchmark","country_code","indicator_group","indicator_code"])

display(cov_df)
cov_path=f"outputs/worldbank_coverage_audit_{sy}_{ey}.csv"
cov_df.to_csv(cov_path,index=False,encoding="utf-8-sig")

print("coverage audit saved:",cov_path)

,country_code,country_name,is_benchmark,indicator_code,indicator_name,indicator_group,total_years,available_years,missing_years_count,missing_years,first_available_year,latest_available_year,coverage_rate
8,IDN,Indonesia,False,IT.CEL.SETS.P2,Mobile cellular subscriptions (per 100 people),DRI,11,10,1,[2025],2015,2024,0.909091
9,IDN,Indonesia,False,IT.NET.BBND.P2,Fixed broadband subscriptions (per 100 people),DRI,11,10,1,[2025],2015,2024,0.909091
10,IDN,Indonesia,False,IT.NET.USER.ZS,Individuals using the Internet (% of population),DRI,11,10,1,[2025],2015,2024,0.909091
11,IDN,Indonesia,False,SL.AGR.EMPL.ZS,Employment in agriculture (% of total employment),EVI,11,11,0,[],2015,2025,1.000000
12,IDN,Indonesia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,11,11,0,[],2015,2025,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3,CHN,China,True,SL.AGR.EMPL.ZS,Employment in agriculture (% of total employment),EVI,11,11,0,[],2015,2025,1.000000
4,CHN,China,True,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,11,11,0,[],2015,2025,1.000000
5,CHN,China,True,SL.SRV.EMPL.ZS,Employment in services (% of total employment),EVI,11,11,0,[],2015,2025,1.000000
6,CHN,China,True,SL.UEM.1524.ZS,Youth unemployment (% of total labor force age...,EVI,11,11,0,[],2015,2025,1.000000


coverage audit saved: outputs/worldbank_coverage_audit_2015_2025.csv


In [36]:
#7.coverage by country
cty_sum=(panel_df.groupby(["country_code","country_name","is_benchmark"])
    .agg(total_data_points=("value","size"),
        available_data_points=("is_missing",lambda x:(~x).sum()),
        missing_data_points=("is_missing","sum")).reset_index())

cty_sum["overall_coverage_rate"]=cty_sum["available_data_points"]/cty_sum["total_data_points"]
cty_sum=cty_sum.sort_values(["is_benchmark","overall_coverage_rate"],
                            ascending=[True,False])

display(cty_sum)
cty_path=f"outputs/worldbank_country_coverage_summary_{sy}_{ey}.csv"
cty_sum.to_csv(cty_path,index=False,encoding="utf-8-sig")

print("country coverage saved:",cty_path)

,country_code,country_name,is_benchmark,total_data_points,available_data_points,missing_data_points,overall_coverage_rate
1,IDN,Indonesia,False,88,85,3,0.965909
2,KHM,Cambodia,False,88,85,3,0.965909
4,MYS,Malaysia,False,88,85,3,0.965909
5,PHL,Philippines,False,88,85,3,0.965909
6,SGP,Singapore,False,88,85,3,0.965909
7,THA,Thailand,False,88,85,3,0.965909
8,VNM,Vietnam,False,88,85,3,0.965909
3,LAO,Laos,False,88,83,5,0.943182
0,CHN,China,True,88,86,2,0.977273


country coverage saved: outputs/worldbank_country_coverage_summary_2015_2025.csv


In [37]:
#8.coverage by year
year_sum=(panel_df.groupby("year")
    .agg(total_data_points=("value","size"),
        available_data_points=("is_missing",lambda x:(~x).sum()),
        missing_data_points=("is_missing","sum")).reset_index())

year_sum["year_coverage_rate"]=year_sum["available_data_points"]/year_sum["total_data_points"]
display(year_sum)
year_path=f"outputs/worldbank_year_coverage_summary_{sy}_{ey}.csv"
year_sum.to_csv(year_path,index=False,encoding="utf-8-sig")

print("year coverage saved:",year_path)

,year,total_data_points,available_data_points,missing_data_points,year_coverage_rate
0,2015,72,72,0,1.000000
1,2016,72,72,0,1.000000
2,2017,72,72,0,1.000000
3,2018,72,72,0,1.000000
4,2019,72,72,0,1.000000
5,2020,72,72,0,1.000000
6,2021,72,72,0,1.000000
7,2022,72,72,0,1.000000
8,2023,72,72,0,1.000000
9,2024,72,70,2,0.972222


year coverage saved: outputs/worldbank_year_coverage_summary_2015_2025.csv


In [38]:
#9.check missing values
miss_df=panel_df[panel_df["is_missing"]==True].copy()
miss_df=miss_df.sort_values(
    ["year","country_code","indicator_group","indicator_code"])

display(miss_df[["year","country_code","country_name","is_benchmark",
    "indicator_group","indicator_code","indicator_name","value"]])

,year,country_code,country_name,is_benchmark,indicator_group,indicator_code,indicator_name,value
691,2024,LAO,Laos,False,DRI,IT.CEL.SETS.P2,Mobile cellular subscriptions (per 100 people),NaN
702,2024,LAO,Laos,False,DRI,IT.NET.BBND.P2,Fixed broadband subscriptions (per 100 people),NaN
780,2025,CHN,China,True,DRI,IT.CEL.SETS.P2,Mobile cellular subscriptions (per 100 people),NaN
791,2025,CHN,China,True,DRI,IT.NET.BBND.P2,Fixed broadband subscriptions (per 100 people),NaN
340,2025,IDN,Indonesia,False,DRI,IT.CEL.SETS.P2,Mobile cellular subscriptions (per 100 people),NaN
351,2025,IDN,Indonesia,False,DRI,IT.NET.BBND.P2,Fixed broadband subscriptions (per 100 people),NaN
329,2025,IDN,Indonesia,False,DRI,IT.NET.USER.ZS,Individuals using the Internet (% of population),NaN
604,2025,KHM,Cambodia,False,DRI,IT.CEL.SETS.P2,Mobile cellular subscriptions (per 100 people),NaN
615,2025,KHM,Cambodia,False,DRI,IT.NET.BBND.P2,Fixed broadband subscriptions (per 100 people),NaN
593,2025,KHM,Cambodia,False,DRI,IT.NET.USER.ZS,Individuals using the Internet (% of population),NaN


In [39]:
#10.formal missing check
asy=2015
aey=2024

formal_miss=panel_df[(panel_df["year"]>=asy)&(panel_df["year"]<=aey)&
 (panel_df["is_missing"]==True)].copy()

formal_miss=formal_miss.sort_values(
    ["year","country_code","indicator_group","indicator_code"])

display(formal_miss[["year","country_code","country_name","is_benchmark",
                    "indicator_group","indicator_code","indicator_name"]])

print("missing data points in 2015-2024:",formal_miss.shape[0])

,year,country_code,country_name,is_benchmark,indicator_group,indicator_code,indicator_name
691,2024,LAO,Laos,False,DRI,IT.CEL.SETS.P2,Mobile cellular subscriptions (per 100 people)
702,2024,LAO,Laos,False,DRI,IT.NET.BBND.P2,Fixed broadband subscriptions (per 100 people)


missing data points in 2015-2024: 2


In [40]:
#11.create formal panel
ana_df=panel_df[(panel_df["year"]>=asy)&(panel_df["year"]<=aey)].copy()
print("formal panel shape:",ana_df.shape)
display(ana_df.head())

ana_path=f"outputs/worldbank_analysis_panel_{asy}_{aey}.csv"
ana_df.to_csv(ana_path,index=False,encoding="utf-8-sig")
print("formal panel saved:",ana_path)

formal panel shape: (720, 10)


,country_code,country_name,is_benchmark,indicator_code,indicator_name,indicator_group,year,value,source,is_missing
0,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2015,27.518841,World Bank API,False
1,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2016,27.486797,World Bank API,False
2,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2017,27.717453,World Bank API,False
3,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2018,27.111039,World Bank API,False
4,MYS,Malaysia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,2019,27.924213,World Bank API,False


formal panel saved: outputs/worldbank_analysis_panel_2015_2024.csv


In [41]:
#12.formal coverage audit
ana_rows=[]
for key,g in ana_df.groupby(grp_cols):
    cc,cname,bench,ind,iname,igrp=key

    miss=list(g.loc[g["is_missing"],"year"].astype(int))
    avail=list(g.loc[~g["is_missing"],"year"].astype(int))

    total_y=len(g)
    avail_y=len(avail)
    miss_y=len(miss)

    ana_rows.append({"country_code":cc,"country_name":cname,
                     "is_benchmark":bench,"indicator_code":ind,
                     "indicator_name":iname,"indicator_group":igrp,
                     "total_years":total_y,"available_years":avail_y,
                     "missing_years_count":miss_y,"missing_years":miss,
        "first_available_year":min(avail) if len(avail)>0 else None,
        "latest_available_year":max(avail) if len(avail)>0 else None,
        "coverage_rate":avail_y/total_y})

ana_cov=pd.DataFrame(ana_rows)
ana_cov=ana_cov.sort_values(
    ["is_benchmark","country_code","indicator_group","indicator_code"])

display(ana_cov)
ana_cov_path=f"outputs/worldbank_coverage_audit_{asy}_{aey}.csv"
ana_cov.to_csv(ana_cov_path,index=False,encoding="utf-8-sig")
print("formal coverage audit saved:",ana_cov_path)

,country_code,country_name,is_benchmark,indicator_code,indicator_name,indicator_group,total_years,available_years,missing_years_count,missing_years,first_available_year,latest_available_year,coverage_rate
8,IDN,Indonesia,False,IT.CEL.SETS.P2,Mobile cellular subscriptions (per 100 people),DRI,10,10,0,[],2015,2024,1.0
9,IDN,Indonesia,False,IT.NET.BBND.P2,Fixed broadband subscriptions (per 100 people),DRI,10,10,0,[],2015,2024,1.0
10,IDN,Indonesia,False,IT.NET.USER.ZS,Individuals using the Internet (% of population),DRI,10,10,0,[],2015,2024,1.0
11,IDN,Indonesia,False,SL.AGR.EMPL.ZS,Employment in agriculture (% of total employment),EVI,10,10,0,[],2015,2024,1.0
12,IDN,Indonesia,False,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,10,10,0,[],2015,2024,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3,CHN,China,True,SL.AGR.EMPL.ZS,Employment in agriculture (% of total employment),EVI,10,10,0,[],2015,2024,1.0
4,CHN,China,True,SL.IND.EMPL.ZS,Employment in industry (% of total employment),EVI,10,10,0,[],2015,2024,1.0
5,CHN,China,True,SL.SRV.EMPL.ZS,Employment in services (% of total employment),EVI,10,10,0,[],2015,2024,1.0
6,CHN,China,True,SL.UEM.1524.ZS,Youth unemployment (% of total labor force age...,EVI,10,10,0,[],2015,2024,1.0


formal coverage audit saved: outputs/worldbank_coverage_audit_2015_2024.csv


In [42]:
#13.create core data
core_inds=["SL.IND.EMPL.ZS","SL.AGR.EMPL.ZS","SL.SRV.EMPL.ZS","SL.UEM.TOTL.ZS",
    "SL.UEM.1524.ZS","IT.NET.USER.ZS","IT.CEL.SETS.P2","IT.NET.BBND.P2"]
supp_inds=[]

core_df=ana_df[ana_df["indicator_code"].isin(core_inds)].copy()
core_df=core_df.sort_values(["country_code","indicator_code","year"])
core_df["value"]=pd.to_numeric(core_df["value"],errors="coerce")
core_df["value_original"]=core_df["value"]
core_df["was_missing_originally"]=core_df["value"].isna()
core_df["value_cleaned"]=(core_df.groupby(["country_code","indicator_code"])["value"]
    .transform(lambda x:x.ffill()))
core_df["was_imputed"]=core_df["was_missing_originally"]&core_df["value_cleaned"].notna()
core_df["impute_method"]="none"
core_df.loc[core_df["was_imputed"],"impute_method"]="ffill"
core_df["missing_position"]="not_missing"
core_df.loc[core_df["was_missing_originally"],"missing_position"]="missing_checked"

# identify whether imputed value is at the end of a time series
core_df["latest_year_in_series"]=(core_df
      .groupby(["country_code","indicator_code"])["year"].transform("max"))
core_df.loc[core_df["was_imputed"]&(core_df["year"]==core_df["latest_year_in_series"]),
    "missing_position"]="end_of_series"
core_df.loc[core_df["was_imputed"]&(core_df["year"]<core_df["latest_year_in_series"]),
    "missing_position"]="middle_of_series"
core_df["imputation_assumption"]="none"
core_df.loc[core_df["was_imputed"],"imputation_assumption"
]="previous available year is used as a conservative proxy"

remain_miss=core_df[core_df["value_cleaned"].isna()]
imp_df=core_df[core_df["was_imputed"]==True].copy()

print("core data shape:",core_df.shape)
print("imputed values:",imp_df.shape[0])
print("remaining missing:",remain_miss.shape[0])

display(core_df.head(20))


if imp_df.shape[0]>0:
    print("imputed rows:")
    display(imp_df[["year","country_code","country_name","indicator_group",
        "indicator_code","indicator_name","value_original","value_cleaned",
        "impute_method","missing_position","imputation_assumption"]])
else:
    print("no imputed values")

final_miss=core_df[core_df["value_cleaned"].isna()].copy()
print("final missing values in core data:",final_miss.shape[0])

if final_miss.shape[0]>0:
    display(final_miss[["year","country_code","country_name","indicator_code",
        "indicator_name","value_original","value_cleaned"]])
else:
    print("core data is complete after cleaning")

exp_core=len(ctys)*len(core_inds)*(aey-asy+1)
act_core=core_df.shape[0]
print("expected core rows:",exp_core)
print("actual core rows:",act_core)
if act_core==exp_core:
    print("core data row number is correct")
else:
    print("check needed: core data row number mismatch")

core data shape: (720, 18)
imputed values: 2
remaining missing: 0


,country_code,country_name,is_benchmark,indicator_code,indicator_name,indicator_group,year,value,source,is_missing,value_original,was_missing_originally,value_cleaned,was_imputed,impute_method,missing_position,latest_year_in_series,imputation_assumption
770,CHN,China,True,IT.CEL.SETS.P2,Mobile cellular subscriptions (per 100 people),DRI,2015,92.540117,World Bank API,False,92.540117,False,92.540117,False,none,not_missing,2024,none
771,CHN,China,True,IT.CEL.SETS.P2,Mobile cellular subscriptions (per 100 people),DRI,2016,97.213877,World Bank API,False,97.213877,False,97.213877,False,none,not_missing,2024,none
772,CHN,China,True,IT.CEL.SETS.P2,Mobile cellular subscriptions (per 100 people),DRI,2017,104.073182,World Bank API,False,104.073182,False,104.073182,False,none,not_missing,2024,none
773,CHN,China,True,IT.CEL.SETS.P2,Mobile cellular subscriptions (per 100 people),DRI,2018,116.229125,World Bank API,False,116.229125,False,116.229125,False,none,not_missing,2024,none
774,CHN,China,True,IT.CEL.SETS.P2,Mobile cellular subscriptions (per 100 people),DRI,2019,122.670392,World Bank API,False,122.670392,False,122.670392,False,none,not_missing,2024,none
775,CHN,China,True,IT.CEL.SETS.P2,Mobile cellular subscriptions (per 100 people),DRI,2020,120.496715,World Bank API,False,120.496715,False,120.496715,False,none,not_missing,2024,none
776,CHN,China,True,IT.CEL.SETS.P2,Mobile cellular subscriptions (per 100 people),DRI,2021,121.491918,World Bank API,False,121.491918,False,121.491918,False,none,not_missing,2024,none
777,CHN,China,True,IT.CEL.SETS.P2,Mobile cellular subscriptions (per 100 people),DRI,2022,124.195249,World Bank API,False,124.195249,False,124.195249,False,none,not_missing,2024,none
778,CHN,China,True,IT.CEL.SETS.P2,Mobile cellular subscriptions (per 100 people),DRI,2023,128.247014,World Bank API,False,128.247014,False,128.247014,False,none,not_missing,2024,none
779,CHN,China,True,IT.CEL.SETS.P2,Mobile cellular subscriptions (per 100 people),DRI,2024,131.753932,World Bank API,False,131.753932,False,131.753932,False,none,not_missing,2024,none


imputed rows:


,year,country_code,country_name,indicator_group,indicator_code,indicator_name,value_original,value_cleaned,impute_method,missing_position,imputation_assumption
691,2024,LAO,Laos,DRI,IT.CEL.SETS.P2,Mobile cellular subscriptions (per 100 people),NaN,64.758807,ffill,end_of_series,previous available year is used as a conservat...
702,2024,LAO,Laos,DRI,IT.NET.BBND.P2,Fixed broadband subscriptions (per 100 people),NaN,2.720211,ffill,end_of_series,previous available year is used as a conservat...


final missing values in core data: 0
core data is complete after cleaning
expected core rows: 720
actual core rows: 720
core data row number is correct


In [43]:
#14.save core data and imputation audit
core_path=f"outputs/worldbank_core_analysis_cleaned_{asy}_{aey}.csv"
core_df.to_csv(core_path,index=False,encoding="utf-8-sig")
print("core data saved:",core_path)

imp_path=f"outputs/worldbank_imputation_audit_{asy}_{aey}.csv"
imp_df.to_csv(imp_path,index=False,encoding="utf-8-sig")
print("imputation audit saved:",imp_path)

core data saved: outputs/worldbank_core_analysis_cleaned_2015_2024.csv
imputation audit saved: outputs/worldbank_imputation_audit_2015_2024.csv


In [44]:
#14.1.core indicator coverage
core_cov=(core_df
    .groupby(["indicator_group","indicator_code","indicator_name"])
    .agg(total_rows=("value_cleaned","size"),
        missing_after_cleaning=("value_cleaned",lambda x:x.isna().sum()),
        imputed_values=("was_imputed",lambda x:x.sum())).reset_index())

core_cov["clean_coverage_rate"]=(
    (core_cov["total_rows"]-core_cov["missing_after_cleaning"])/
    core_cov["total_rows"])

display(core_cov)
core_cov_path=f"outputs/worldbank_core_indicator_coverage_{asy}_{aey}.csv"
core_cov.to_csv(core_cov_path,index=False,encoding="utf-8-sig")
print("core indicator coverage saved:",core_cov_path)

,indicator_group,indicator_code,indicator_name,total_rows,missing_after_cleaning,imputed_values,clean_coverage_rate
0,DRI,IT.CEL.SETS.P2,Mobile cellular subscriptions (per 100 people),90,0,1,1.0
1,DRI,IT.NET.BBND.P2,Fixed broadband subscriptions (per 100 people),90,0,1,1.0
2,DRI,IT.NET.USER.ZS,Individuals using the Internet (% of population),90,0,0,1.0
3,EVI,SL.AGR.EMPL.ZS,Employment in agriculture (% of total employment),90,0,0,1.0
4,EVI,SL.IND.EMPL.ZS,Employment in industry (% of total employment),90,0,0,1.0
5,EVI,SL.SRV.EMPL.ZS,Employment in services (% of total employment),90,0,0,1.0
6,EVI,SL.UEM.1524.ZS,Youth unemployment (% of total labor force age...,90,0,0,1.0
7,EVI,SL.UEM.TOTL.ZS,"Unemployment,total (% of total labor force)",90,0,0,1.0


core indicator coverage saved: outputs/worldbank_core_indicator_coverage_2015_2024.csv


In [45]:
#15.data decision summary
from datetime import datetime
now=datetime.now()
run_timestamp=now.strftime("%Y-%m-%d %H:%M:%S")
run_id=now.strftime("%Y%m%d_%H%M%S")
print("run timestamp:",run_timestamp)
print("run id:",run_id)

y2025_rate=year_sum.loc[year_sum["year"]==2025,"year_coverage_rate"].iloc[0]
y2025_avail=year_sum.loc[year_sum["year"]==2025,"available_data_points"].iloc[0]
y2025_total=year_sum.loc[year_sum["year"]==2025,"total_data_points"].iloc[0]

formal_miss_n=formal_miss.shape[0]
imp_n=imp_df.shape[0]
remain_n=remain_miss.shape[0]
if imp_n>0:
    imp_lines=[]
    for i,r in imp_df.iterrows():
        imp_lines.append(
            f"- {int(r['year'])},{r['country_name']},{r['indicator_code']},{r['indicator_name']},method={r['impute_method']}"
        )
    imp_txt="\n".join(imp_lines)
else:
    imp_txt="No missing values were imputed in the core dataset."

if remain_n>0:
    remain_lines=[]
    for i,r in remain_miss.iterrows():
        remain_lines.append(
            f"- {int(r['year'])},{r['country_name']},{r['indicator_code']},{r['indicator_name']}"
        )
    remain_txt="\n".join(remain_lines)
else:
    remain_txt="No missing values remain in the core dataset after cleaning."

decision_txt=f"""
Data decision summary

Generated time:
{run_timestamp}

Run ID:
{run_id}

Raw download period:
{sy}-{ey}

Formal analysis period:
{asy}-{aey}

Countries:
8 Southeast Asian countries were used as the main sample.
China was included as an external benchmark only.

Reason for excluding 2025:
The year-level data coverage rate for 2025 was {y2025_rate:.2%} ({int(y2025_avail)}/{int(y2025_total)} available data points).
Therefore, 2025 was downloaded but excluded from the formal analysis.

Formal missing values:
There were {formal_miss_n} missing data points in the {asy}-{aey} formal analysis panel before indicator selection and cleaning.

EVI input indicators kept in the core dataset:
1.Employment in industry
2.Employment in agriculture
3.Employment in services
4.Total unemployment
5.Youth unemployment

DRI input indicators kept in the core dataset:
1.Internet users
2.Mobile cellular subscriptions
3.Fixed broadband subscriptions

DRI indicator decision:
No DRI indicator was excluded from the core dataset. Fixed broadband subscriptions were retained in the core DRI
because the formal analysis panel only had two end-of-series DRI missing values for Laos in 2024,
and both were handled using forward fill.

Missing value treatment:
Forward fill was applied within each country-indicator time series after sorting by country,indicator and year.
This rule was applied to the core dataset, including the service employment indicator kept for the EVI robustness version.

Limitation of forward fill:
Forward fill assumes that the latest available value can be used as a conservative proxy for the missing year.
This assumption is more acceptable for a small number of end-of-series missing values, but it may be less accurate
for middle-of-series gaps or indicators that change rapidly.
Therefore, all imputed values were flagged using was_imputed and recorded in the imputation audit file.

Imputed values:
{imp_txt}

Remaining missing values:
{remain_txt}

China treatment:
China can be shown in index comparison and visualisation.
China should not be included in Southeast Asia regional average or K-means clustering.
"""

print(decision_txt)

decision_path="outputs/data_decision_summary.txt"

with open(decision_path,"w",encoding="utf-8") as f:
    f.write(decision_txt)

print("data decision summary saved:",decision_path)

meta_txt=f"""
Run metadata

Run ID:
{run_id}

Generated time:
{run_timestamp}

Raw data period:
{sy}-{ey}

Formal analysis period:
{asy}-{aey}

Country number:
{len(ctys)}

Indicator number:
{len(inds)}

Raw data rows:
{raw_df.shape[0]}

Core data rows:
{core_df.shape[0]}

Failed API requests:
{fail_df.shape[0]}

Duplicate raw rows:
{dup_df.shape[0]}

Formal missing values before cleaning:
{formal_miss.shape[0]}

Imputed values:
{imp_df.shape[0]}

Remaining missing values:
{remain_miss.shape[0]}

Core data file:
outputs/worldbank_core_analysis_cleaned_{asy}_{aey}.csv

Decision summary file:
outputs/data_decision_summary.txt
"""

meta_path="outputs/run_metadata.txt"

with open(meta_path,"w",encoding="utf-8") as f:
    f.write(meta_txt)
print("run metadata saved:",meta_path)

run timestamp: 2026-06-17 13:30:21
run id: 20260617_133021

Data decision summary

Generated time:
2026-06-17 13:30:21

Run ID:
20260617_133021

Raw download period:
2015-2025

Formal analysis period:
2015-2024

Countries:
8 Southeast Asian countries were used as the main sample.
China was included as an external benchmark only.

Reason for excluding 2025:
The year-level data coverage rate for 2025 was 63.89% (46/72 available data points).
Therefore, 2025 was downloaded but excluded from the formal analysis.

Formal missing values:
There were 2 missing data points in the 2015-2024 formal analysis panel before indicator selection and cleaning.

EVI input indicators kept in the core dataset:
1.Employment in industry
2.Employment in agriculture
3.Employment in services
4.Total unemployment
5.Youth unemployment

DRI input indicators kept in the core dataset:
1.Internet users
2.Mobile cellular subscriptions
3.Fixed broadband subscriptions

DRI indicator decision:
No DRI indicator was exclud

In [46]:
#16.check saved files
from pathlib import Path

print("raw-data files:")
for f in Path("raw-data").glob("*"):
    print(f)

print("outputs files:")
for f in Path("outputs").glob("*"):
    print(f)

raw-data files:
raw-data/worldbank_raw_api_data_2015_2025.csv
outputs files:
outputs/worldbank_core_indicator_coverage_2015_2024.csv
outputs/worldbank_panel_2015_2025.csv
outputs/run_metadata.txt
outputs/worldbank_coverage_audit_2015_2025.csv
outputs/worldbank_coverage_audit_2015_2024.csv
outputs/worldbank_core_analysis_cleaned_2015_2024.csv
outputs/worldbank_analysis_panel_2015_2024.csv
outputs/data_decision_summary.txt
outputs/worldbank_country_coverage_summary_2015_2025.csv
outputs/worldbank_year_coverage_summary_2015_2025.csv
outputs/worldbank_imputation_audit_2015_2024.csv


In [47]:
#17.mount drive
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [48]:
#18.create drive folder
from pathlib import Path

drive_dir=Path("/content/drive/MyDrive/STQD6324_Labour_Shock")
raw_drive=drive_dir/"raw-data"
out_drive=drive_dir/"outputs"

raw_drive.mkdir(parents=True,exist_ok=True)
out_drive.mkdir(parents=True,exist_ok=True)

print("drive folder ready:",drive_dir)

drive folder ready: /content/drive/MyDrive/STQD6324_Labour_Shock


In [51]:
#19.copy files to drive
import shutil
from pathlib import Path

for f in Path("raw-data").glob("*"):
    shutil.copy(f,raw_drive/f.name)

for f in Path("outputs").glob("*"):
    shutil.copy(f,out_drive/f.name)

print("files copied to drive")

files copied to drive


In [52]:
#20.check drive files
print("drive raw-data files:")
for f in raw_drive.glob("*"):
    print(f)

print("drive outputs files:")
for f in out_drive.glob("*"):
    print(f)

drive raw-data files:
/content/drive/MyDrive/STQD6324_Labour_Shock/raw-data/worldbank_raw_api_data_2015_2025.csv
drive outputs files:
/content/drive/MyDrive/STQD6324_Labour_Shock/outputs/worldbank_core_indicator_coverage_2015_2024.csv
/content/drive/MyDrive/STQD6324_Labour_Shock/outputs/worldbank_panel_2015_2025.csv
/content/drive/MyDrive/STQD6324_Labour_Shock/outputs/worldbank_coverage_audit_2015_2025.csv
/content/drive/MyDrive/STQD6324_Labour_Shock/outputs/worldbank_coverage_audit_2015_2024.csv
/content/drive/MyDrive/STQD6324_Labour_Shock/outputs/worldbank_core_analysis_cleaned_2015_2024.csv
/content/drive/MyDrive/STQD6324_Labour_Shock/outputs/worldbank_analysis_panel_2015_2024.csv
/content/drive/MyDrive/STQD6324_Labour_Shock/outputs/data_decision_summary.txt
/content/drive/MyDrive/STQD6324_Labour_Shock/outputs/worldbank_country_coverage_summary_2015_2025.csv
/content/drive/MyDrive/STQD6324_Labour_Shock/outputs/worldbank_year_coverage_summary_2015_2025.csv
/content/drive/MyDrive/STQD